3. abalone.txt
   1. abalone_minimax.py로 파일 만들것
   2. 표준라이브러리 pands사용가능 scikitlearn금지
   3. 입출력/인코딩 파일경로 고정
   4. load_attributes(), load_data(), make_label(df), min_max_manual(df) 등으로 함수 분리
   5. 상수열(분모0) 에외처리 로직 포함)
   6. 출력규칙, 위 3개 print()수행 부가 출력 금지, 문자열은 "" , 
   7. zip을 다운 받아 풀면 
   8. abalone data.txt? data 수치만 들어 있음 컴마로 분리됨, 2, M, 0.35, 0.265  ..................
   9. abalone_attribute.txt 헤더가 들어 있음 
abalone_attributes.txt
Sex
Length
Diameter
Height
Whole weight
Shucked weight
Viscera weight
Shell weight
Rings

abalone.txt
M,0.455,0.365,0.095,0.514,0.2245,0.101,0.15,15
M,0.35,0.265,0.09,0.2255,0.0995,0.0485,0.07,7
F,0.53,0.42,0.135,0.677,0.2565,0.1415,0.21,9
M,0.44,0.365,0.125,0.516,0.2155,0.114,0.155,10
생략

   10. 제공된 두 파일을 읽어 하나의 dataframe으로 구성. 
   11. 전복은 유아기에는 성별이 정해지지 않다가 성장하면서 성별이 정해지는 특성으로 sex컬럼을 별도로 label 컬럼으로 분리후 원본 sex컬럼제거.
   12. 이후 label제외한 모든 수치컬럼에 대해 min max scaling을 직접 수식으로 구현해서 적용
   13. 요구사항 충족하는 프로그램 작성
   14. 데이터 적재
   15. 현재 작업 디렉토리의 abalone_sttribte.txt에서 열이름 목록으로 읽어 (행단위 텍스트->리스트) 같은 디렉토리의 abalon.txt 콤마를 읽어 pandas.dataframe을 생성
   16. 앞서 읽은 열이름을 컴럼명으로 적용
   17. 라벨분리, sex컬럼을 그대로 복사 label컬럼을 만들고 [예 'M','F'] 원본 sex컴럼은 제거한다.
   18. 결과 table에는 label만 남김
   19. 스케일링 대상/ 방법
   20. label을 제외한 모든 수치컬럼에 대해 열별로 min-max scaling (직접수식)을 적용한다
   21. x'=max(x)-min(x)x-min(x)
   22. x'' = x-min(x)max(x)-min(x)
   23. x' = \frac{x-\min(x)|{\max(x)-\min(x)}}
   24. 분모가 0(상수열)인 경우 해달열을 0.0 으로 채워줌
   25. 출력형식 print() 총 3번
   26. 원본 data frame 모양 df.shape <-예(N,9)
   27. 라벨 분포 label value-counts().to_dict()
   28. 스케일결과의 상,하한요약 
   29. scaled.describe().loc[[min,max]].round(6).to_dict]
   30. 대상열들의 상,하한 요약
   31. scaled.describe().loc[[min,max]].round(6).to_dict]
   32. (대상열들의 min=0.0, max=1.0임을 확인)
   33. 예외처리 (우선순위)
       1.  1. 파일열기 실패 File open error
       2.  2. 디코딩 에러 Decoding error
       3.  3. 피싱/스케일링 단계, 기타 오류 - processing error
       4.  각 경우 정확한 문자열을 print()하고 return 으로 종료 (sys,exit()금지)
   34. 다운 받아서 풀면 다음 파일이 있다. 
   35. 

좋아요 👍 완벽하게 이해하려면 지금 과제 코드에서 사용한 **pandas 문법**을 한 줄씩 뜯어보는 게 가장 좋습니다.
아래는 `abalone_minimax.py`에서 등장한 주요 pandas 관련 문법 정리예요.
(단순 함수 호출이 아니라 “무엇을 어떻게 하는가 + 내부 동작”까지 설명합니다.)

---

## 🧱 1. `import pandas as pd`

> pandas 라이브러리를 불러와서 `pd`라는 짧은 이름으로 사용하겠다는 뜻입니다.
> Python 관례적으로 pandas는 항상 `pd`로 import합니다.

```python
import pandas as pd
```

---

## 🧾 2. `pd.read_csv(path, names=columns, header=None)`

> CSV(콤마 구분) 형식의 파일을 읽어서 **DataFrame**으로 만들어주는 함수입니다.

* `path`: 읽을 파일 경로 (`"abalone.txt"`)
* `names=columns`: 파일에 헤더가 없으므로, 우리가 따로 읽어온 `abalone_attributes.txt`의 내용을 **컬럼 이름으로 지정**
* `header=None`: 파일의 첫 줄을 컬럼명으로 사용하지 말고, 그냥 데이터로 처리

📌 결과:
각 행(row)은 데이터 1개, 각 열(column)은 속성 하나를 갖는 **표(DataFrame)** 객체 생성

```python
df = pd.read_csv("abalone.txt", names=cols, header=None)
```

---

## 🧍‍♀️ 3. `df.copy()`

> 원본 `DataFrame`을 건드리지 않고, 완전히 **복사본**을 만듭니다.
> 복사하지 않고 직접 수정하면 원본 `df`도 바뀌므로 안전하게 사본을 만들어 작업할 때 사용합니다.

```python
df = df.copy()
```

---

## 🧩 4. `df["label"] = df["Sex"]`

> 기존 컬럼 `"Sex"`를 그대로 복사해서 `"label"`이라는 새 컬럼을 만듭니다.

즉,

```
Sex   →  label (복사)
```

이후 `"Sex"`는 삭제합니다.

---

## 🧹 5. `df.drop(columns=["Sex"])`

> DataFrame에서 `"Sex"` 컬럼을 제거합니다.

* `columns` 인자는 여러 열을 리스트로 받을 수 있습니다.
* drop()은 새로운 객체를 반환하므로, 다시 `df = ...` 형태로 재할당합니다.

```python
df = df.drop(columns=["Sex"])
```

---

## 🔢 6. `df[col].min()` / `df[col].max()`

> 특정 컬럼의 **최솟값/최댓값**을 구합니다.
> Min-Max 스케일링의 기본 자료로 사용합니다.

```python
col_min = df[col].min()
col_max = df[col].max()
```

---

## ⚖️ 7. `(df[col] - col_min) / (col_max - col_min)`

> **Min-Max Scaling 공식**을 수식 그대로 pandas Series(열 단위)에 적용한 것.
> 각 값이 `0~1` 범위로 변환됩니다.

pandas는 **벡터 연산**을 지원하기 때문에, for문 없이도 컬럼 단위 연산이 빠르게 동작합니다.

---

## 🚫 8. `scaled[col] = 0.0`

> 만약 `(max == min)`이면 모든 값이 같아서 분모가 0이 됩니다.
> 그럴 땐 해당 열을 전부 `0.0`으로 채웁니다.

---

## 🔍 9. `label.value_counts().to_dict()`

> 각 label(‘M’, ‘F’, ‘I’)이 몇 번 나왔는지를 **빈도표**로 계산합니다.

* `value_counts()` → Series:

  ```
  M    1528
  I    1342
  F    1307
  dtype: int64
  ```
* `to_dict()` → 일반 Python 딕셔너리 변환

  ```
  {'M': 1528, 'I': 1342, 'F': 1307}
  ```

---

## 📊 10. `scaled.describe()`

> 수치형 컬럼들의 통계 요약표를 자동으로 만들어줍니다.
> 기본 8개 항목(`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`).

예시:

```
              Length    Diameter    Height ...
count   4177.000000  4177.000000  4177.000000
mean       0.487...,  ...
min        0.000000  ...
max        1.000000  ...
```

---

## 🧭 11. `.loc[["min", "max"]]`

> `describe()` 결과에서 **행 인덱스가 “min”, “max”인 부분만 선택**합니다.
> 즉, 각 컬럼의 최소값/최대값만 요약해 보기 좋게 출력합니다.

---

## 🔢 12. `.round(6)`

> 소수점 6자리까지 반올림합니다.
> 평가 과제에서는 출력을 깔끔히 맞추기 위해 사용합니다.

---

## 🧮 13. `.to_dict()`

> DataFrame이나 Series를 **딕셔너리(dict)** 형태로 변환합니다.
> `scaled.describe().loc[["min", "max"]].round(6).to_dict()`
> → 각 컬럼 이름별로 {‘min’: 값, ‘max’: 값} 형태의 딕셔너리 생성.

예:

```python
{'Length': {'min': 0.0, 'max': 1.0}, ...}
```

---

## 🧠 요약 구조

```python
import pandas as pd

cols = load_attributes()  # 헤더 읽기
df = pd.read_csv(DATA_PATH, names=cols, header=None)  # 파일 읽기
df["label"] = df["Sex"]
df = df.drop(columns=["Sex"])
scaled = (df - df.min()) / (df.max() - df.min())  # Min-Max
scaled.describe().loc[["min","max"]].round(6).to_dict()
```

---

원하신다면 위 문법 중
예를 들어 `describe()`, `loc[]`, `value_counts()` 같은 것들을
**pandas 내부 구조와 함께** (예: 어떤 자료형 반환하는지, 어떻게 인덱싱되는지)
한 단계 더 깊이 “전문가 수준으로” 설명해 드릴 수도 있습니다.
그렇게 해드릴까요?


좋아요 — 자세히, 단계별로 깊게 설명할게요.
아래 내용은 **`abalone_minimax.py`**에서 사용한 주요 `pandas` 문법과 그 내부 동작·반응형 예제·주의점·모범 사용법까지 포함한 전문가 수준 설명입니다. 코드 예시는 한국어 주석과 함께 제공하고, 실무에서 흔히 마주치는 함정(에러·성능·복사·dtype 이슈)도 다룹니다.

# 1) 기본 개념 정리 — DataFrame vs Series

* `DataFrame` : 2차원 표(행 × 열). `pd.DataFrame` 객체. 여러 컬럼(Series)을 묶은 것.
* `Series` : 1차원 배열(인덱스 + 값). `df["col"]` 또는 `df.col`은 Series 반환.

자료형(dtypes)은 각 열별로 다를 수 있음 (`int64`, `float64`, `object` 등). 숫자가 문자열로 들어오면 연산이 안 되므로 `read_csv`에서 `dtype` 지정하거나 `pd.to_numeric()`로 변환해야 합니다.

---

# 2) 파일 읽기: `pd.read_csv(path, names=cols, header=None)`

* 기능 : CSV를 읽어 `DataFrame` 생성.
* 핵심 파라미터

  * `names=cols` : 파일에 헤더가 없을 때 외부에서 컬럼명 지정.
  * `header=None` : 파일 첫 줄을 컬럼명으로 쓰지 않음.
  * `encoding` : 디코딩 문제가 있으면 `encoding="utf-8"` 또는 `"latin1"` 등 지정.
  * `dtype` : 컬럼별 타입 강제 지정 가능 → 읽기 후의 타입 안정화에 도움.
* 예외: 파일 없으면 `FileNotFoundError`, 디코딩 문제면 `UnicodeDecodeError`, 내부 파싱 오류면 `pd.errors.ParserError` 등 발생 가능.

```python
cols = ["Sex","Length","Diameter","Height","Whole weight","Shucked weight","Viscera weight","Shell weight","Rings"]
df = pd.read_csv("abalone.txt", names=cols, header=None, encoding="utf-8")
```

---

# 3) 열 추가·복사: `df["label"] = df["Sex"]`

* 동작: `df["Sex"]` (Series)을 읽어 동일한 값으로 새 컬럼 `label` 생성.
* 내부: 새 `Series`가 `DataFrame`에 삽입됨.
* 주의: `df["new"] = something`은 안전한 패턴. 하지만 슬라이싱 결과에 대해 바로 할당하면 `SettingWithCopyWarning` 발생 가능 — 아래에서 설명.

---

# 4) 열 삭제: `df.drop(columns=["Sex"])`

* `df.drop(...)`은 기본적으로 **새 DataFrame**을 반환(원본 보존).
* `inplace=True`로 하면 원본 수정(비추천: 가독성/디버깅에 불리).
* 권장: `df = df.drop(columns=["Sex"])`로 명시적 재할당.

```python
df = df.drop(columns=["Sex"])   # 안전한 패턴
```

---

# 5) 복사와 `SettingWithCopyWarning`: `df.copy()`

* `df.copy()`는 **완전한 복사본**(deep copy)을 만듦. 원본과 독립적.
* 왜 필요한가? 예:

  ```python
  sub = df[df['Rings'] > 10]   # sub는 원래 df의 view(또는 ambiguous copy)일 수 있음
  sub['Length'] = sub['Length'] * 2   # 경고: SettingWithCopyWarning
  ```
* 안전한 방식:

  ```python
  sub = df[df['Rings'] > 10].copy()
  sub['Length'] = sub['Length'] * 2   # 경고 없음
  ```

---

# 6) 통계: `df[col].min()`, `df[col].max()`, `df.min()`, `df.max()`

* `Series.min()`/`max()` : 해당 열의 최소/최댓값 (NaN은 기본 제외).
* `df.min()` : 각 열별 최소값을 가지는 Series 반환 (열 인덱스가 인덱스가 됨).
* 벡터화 연산에 유리: `df - df.min()`는 **열별로** (브로드캐스트) 최소값을 뺌.

```python
mins = df.min()            # 각 컬럼별 최소값 Series
maxs = df.max()            # 각 컬럼별 최대값 Series
scaled = (df - mins) / (maxs - mins)  # 자동으로 컬럼별 브로드캐스트됨
```

* 주의: `maxs - mins`의 어떤 원소가 0이면 분모 0 → `inf` 또는 `NaN`. 처리 필요.

---

# 7) 벡터화와 브로드캐스트(성능 중요)

* pandas는 내부적으로 numpy 배열 연산을 사용하므로 **컬럼 전체를 한 번에 연산**하면 파이썬 for-loop보다 훨씬 빠릅니다.
* 따라서 Min–Max 스케일링은 컬럼별 반복 대신 가능한 한 벡터화 방식으로:

  ```python
  mins = data.min()
  maxs = data.max()
  denom = (maxs - mins).replace(0, 1)  # 분모를 0 -> 1로 바꿔서 0나누기 방지
  scaled = (data - mins) / denom
  # 이후 상수열이었던 칼럼들은 수동으로 0.0 채우기 처리
  ```

---

# 8) 안전한 Min–Max 스케일 구현 (분모 0 처리)

* 상수열(모든 값 동일) 처리 규칙(요구사항): 그 열 전체를 `0.0`으로 채움.
* 안전 구현 예:

```python
mins = data.min()
maxs = data.max()
denom = maxs - mins

scaled = data.copy()
for col in data.columns:
    if denom[col] == 0:
        scaled[col] = 0.0
    else:
        scaled[col] = (data[col] - mins[col]) / denom[col]
```

* 벡터화로 처리하려면:

```python
denom = maxs - mins
safe_denom = denom.replace(0, 1)   # 0을 1로 바꿔 나눗셈 실행
scaled = (data - mins) / safe_denom
zero_cols = denom[denom == 0].index
scaled[zero_cols] = 0.0
```

---

# 9) 결측치(NaN) 처리: `pd.isna()` / `.fillna()` / `dropna()`

* `df.min()`/`max()`는 기본적으로 NaN을 무시. 하지만 열 전체가 NaN이면 결과는 NaN → `isna` 체크 필요.
* 스케일링 전 결측치 확인:

  ```python
  df.isna().sum()   # 컬럼별 NaN 개수
  ```
* NaN 대응: `df.fillna(value)` 또는 전/후에 처리(예: `scaled.fillna(0.0)`).

---

# 10) value_counts()와 to_dict()

* `label.value_counts()` : Series 반환. 인덱스는 라벨(예: 'M','F','I'), 값은 빈도수.
* `.to_dict()` : `{label: count, ...}` 형태의 파이썬 dict 반환.
* 예: `{'M': 1528, 'I': 1342, 'F': 1307}`

---

# 11) describe()와 인덱싱: `scaled.describe().loc[["min", "max"]].round(6).to_dict()`

* `scaled.describe()` : 각 수치형 컬럼에 대해 `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`의 행을 갖는 `DataFrame` 반환.

  * 반환 형태: index는 통계명(`count`, `mean`, ...), columns는 원래 컬럼명.
* `.loc[["min","max"]]` : 그 결과에서 행 `min`과 `max`만 선택 → `DataFrame` (2 × ncols).
* `.round(6)` : 소수점 6자리 반올림.
* `.to_dict()` : 열별로 dictionary 구조화됨. 결과 구조 예:

  ```python
  {
    'Length': {'min': 0.0, 'max': 1.0},
    'Diameter': {'min': 0.0, 'max': 1.0},
    ...
  }
  ```
* 주의: `.to_dict()` 기본 방향은 `dict[ column_name ] = { index_name: value }`.

---

# 12) 색인(Indexing) : `.loc[]` vs `.iloc[]`

* `.loc[row_label, col_label]` : 라벨 기반 색인(인덱스/컬럼명이 사용될 때).
* `.iloc[row_pos, col_pos]` : 정수 위치 기반 색인.
* 예:

  ```python
  df.loc[0, "Length"]   # 0번 인덱스의 Length 값
  df.iloc[0, 1]         # 0행 1열 (정수 인덱스)
  ```
* 복수 선택: `df.loc[:, ["Length", "Diameter"]]` (모든 행의 두 컬럼).

---

# 13) `shape`, `head()`, `columns`

* `df.shape` : `(행수, 열수)` 튜플.
* `df.head(n=5)` : 상위 n행 반환(디버깅에 유용).
* `df.columns` : 컬럼명 Index 반환.

---

# 14) 예외 처리(파싱/읽기)와 pandas

* `pd.read_csv()` 내부에서 발생 가능한 예외들:

  * `FileNotFoundError` : 파일 경로가 잘못된 경우.
  * `UnicodeDecodeError` : 인코딩 불일치.
  * `pd.errors.ParserError` : 파일 형식이 불규칙할 때.
* 스크립트 레벨에서 `try/except`로 분류하여 사용자 지정 메시지 출력(요구사항대로) 처리.

---

# 15) 성능 팁 (대규모 데이터에서 중요)

* 벡터화 사용: 컬럼 단위 연산을 이용하라(반복문 피함).
  `for row in df.itertuples()` 처럼 한 행씩 순회하는 방법은 매우 느림.
* 메모리: `df.astype({'Rings':'int32'})` 등으로 dtype을 줄이면 메모리 절약.
* 필요한 컬럼만 읽기: `pd.read_csv(..., usecols=[...])`.
* `.apply()`도 파이썬 함수를 각 항목에 반복 실행하므로 느릴 수 있음. numpy 벡터화가 더 빠름.
* 큰 데이터는 chunked reading (`pd.read_csv(..., chunksize=10000)`)로 처리.

---

# 16) 흔한 실수와 디버깅 팁

* `SettingWithCopyWarning` : 슬라이싱 뒤에 바로 할당하려 할 때 발생. 해결: `.copy()` 사용 or `.loc`를 사용해 명시적 할당.

  ```python
  df.loc[df['Rings'] > 10, 'Length'] = df['Length'] * 2  # 안전
  ```
* 분모 0 처리 안 함 → `inf`/`NaN` 발생. 항상 체크.
* 문자열 컬럼을 숫자로 연산하려 하면 `TypeError` 또는 unexpected 결과. `pd.to_numeric(..., errors='coerce')`로 강제 변환(변환 실패는 NaN).
* `read_csv`에서 `names=...`와 `header=None`을 동시에 사용하면 파일 첫 줄을 데이터로 읽음(원하는 동작이면 OK).

---

# 17) 실무 예시: 안전한 스케일링 함수 (완결 코드)

```python
def safe_min_max_scale(df):
    # df: 숫자형 컬럼만 포함한 DataFrame
    mins = df.min()
    maxs = df.max()
    denom = maxs - mins

    # 분모가 0인 열(상수열) 인덱스
    zero_cols = denom[denom == 0].index

    # 분모 0을 일시적으로 1로 바꿔 안전하게 나눔
    safe_denom = denom.replace(0, 1)
    scaled = (df - mins) / safe_denom

    # 상수열은 요구사항대로 0.0으로 채움
    if len(zero_cols) > 0:
        scaled[zero_cols] = 0.0

    return scaled
```

---

# 18) `describe()` 결과 구조 예시(직접 해석)

* `scaled.describe()`가 반환하는 DataFrame 예시(요약):

  ```
         Length   Diameter  Height ...
  count 4177.0    4177.0    4177.0
  mean   0.48     0.47      0.12
  std    0.25     0.24      0.08
  min    0.00     0.00      0.00
  25%    ...
  50%    ...
  75%    ...
  max    1.00     1.00      1.00
  ```
* `.loc[["min","max"]]`로 행 슬라이스 하면 2×ncols DataFrame.
* `.to_dict()` 결과는 열 중심으로 `{col: {stat: value, ...}}`.

---

# 19) 간단한 체크리스트 (코드 짤 때)

1. `read_csv()` : `encoding`, `names`, `header=None` 확인
2. `df.copy()` : 슬라이스 후 수정 시 복사 사용
3. dtype 확인: `df.dtypes` → 숫자형인지 확인
4. NaN 체크: `df.isna().sum()`
5. denom 0 체크 후 스케일링
6. 결과 검증: `scaled.describe().loc[['min','max']]` → 0/1 확인
7. 출력은 규칙(3번) 맞추기

---

# 20) 요약(짧게)

* pandas는 벡터화, 브로드캐스트, 다양한 편리한 통계/인덱싱 기능을 제공.
* `read_csv`, `df[col]`, `df.drop`, `df.copy`, `value_counts`, `describe`, `loc` 등은 매우 자주 사용되는 기본 문법.
* 주된 주의점: `SettingWithCopyWarning`, 분모 0, 결측치, dtype 불일치, 성능(벡터화 권장).

---

원하면 지금 본 내용을 바탕으로:

* `abalone_minimax.py`의 각 줄에 **왜 그런 pandas 문법을 썼는지** 주석을 더 달아 드리거나,
* `safe_min_max_scale()`를 `pandas`만으로 벡터화 최적화한 버전(한 줄 연산)에 대해 성능 비교(시간 측정)를 해 드릴게요.

어떤 걸 먼저 해볼까요?


(base) C:\Users\user\git\codyssey\evaluation>C:\ProgramData\anaconda3\python.exe c:/Users/user/git/codyssey/evaluation/abalone_evaluation.py
(4177, 9)
{'M': 1528, 'I': 1342, 'F': 1307}
{'Length': {'min': 0.0, 'max': 1.0}, 'Diameter': {'min': 0.0, 'max': 1.0}, 'Height': {'min': 0.0, 'max': 1.0}, 'Whole weight': {'min': 0.0, 'max': 1.0}, 'Shucked weight': {'min': 0.0, 'max': 1.0}, 'Viscera weight': {'min': 0.0, 'max': 1.0}, 'Shell weight': {'min': 0.0, 'max': 1.0}, 'Rings': {'min': 0.0, 'max': 1.0}}

👍 네, 아주 잘 나온 **정상적인 결과**입니다!
하나씩 확인해볼게요 👇

---

### ✅ 1️⃣ `(4177, 9)`

➡️ `abalone.txt` 데이터에는 **4177개의 샘플**, **9개의 컬럼**이 있다는 뜻입니다.
이는 원본 UCI Abalone 데이터셋과 동일합니다.

* 1개는 `Sex` (성별)
* 8개는 수치형 특성 (`Length` ~ `Rings`)

즉, 파일을 제대로 읽어왔다는 뜻이에요.

---

### ✅ 2️⃣ `{'M': 1528, 'I': 1342, 'F': 1307}`

➡️ `label` (성별) 컬럼의 **라벨 분포**입니다.

* M (Male): 1528개
* I (Infant): 1342개
* F (Female): 1307개

이것도 실제 UCI Abalone 데이터의 정확한 분포예요.
즉, **성별 컬럼을 label로 잘 분리**한 것입니다.

---

### ✅ 3️⃣ `{'Length': {'min': 0.0, 'max': 1.0}, ...}`

➡️ 모든 수치형 컬럼의 **최소값이 0.0**, **최대값이 1.0**이 되었네요.
이건 **Min–Max Scaling이 정확히 작동**했다는 의미입니다.

즉,
[
X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}
]
공식대로 잘 구현된 결과입니다.

---

### 💯 결론

모든 출력은 완벽합니다.

* 파일 읽기 ✅
* 라벨 분리 ✅
* Min–Max Scaling 구현 ✅
* 결과 검증 ✅

---

원하신다면 다음 단계로

* **`Standard Scaling` (평균 0, 표준편차 1)** 적용
* **Over/Under Sampling 실험**
* **SMOTE 대안 구현 (보너스)**

을 이어서 해드릴까요?
